In [16]:
import pandas as pd
from pymongo import MongoClient

# 1. Conexión a MongoDB
client = MongoClient('mongodb://mongodb:27017')
db = client['steam_db']

print("Exportando datos originales...")

# 2. CARGAR HISTÓRICO (Kaggle)
# Traemos los datos tal cual están en la base de datos
df_hist = pd.DataFrame(list(db.juegos_historicos.find({}, {"_id": 0})))

# Solo hacemos una limpieza básica de nombres para que el buscador de PowerBI funcione bien
df_hist['Name'] = df_hist['Name'].astype(str).str.strip()
df_hist = df_hist.drop_duplicates(subset=['Name'])

# 3. CARGAR REVIEWS (Scraping)
df_rev = pd.DataFrame(list(db.reviews_raw.find({}, {"_id": 0})))

# Mantenemos el mapeo de países porque es lo que hace que el Mapa funcione
mapa_paises = {
    'english': 'United States', 'schinese': 'China', 'russian': 'Russia',
    'spanish': 'Spain', 'brazilian': 'Brazil', 'german': 'Germany',
    'french': 'France', 'japanese': 'Japan', 'koreana': 'South Korea',
    'turkish': 'Turkey', 'polish': 'Poland'
}
df_rev['pais_real'] = df_rev['language'].map(mapa_paises)

# 4. EXPORTACIÓN FINAL PARA POWER BI (Español)
# Usamos decimal=',' para que no tengas que pelearte con los puntos y las comas
df_hist.to_csv('/home/jovyan/work/historico_limpio.csv', index=False, quoting=1, decimal=',')
df_rev.to_csv('/home/jovyan/work/reviews_limpio.csv', index=False, quoting=1, decimal=',')

print(f"Generados {len(df_hist)} juegos y {len(df_rev)} reviews.")

Exportando datos originales (Modo Simple)...
Generados 121455 juegos y 47914 reviews.
